# RAG の失敗を検索と生成に切り分けて分析する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/22_rag/rag_error_analysis.ipynb)

NLP クックブック 22-11 ／ RAG (検索拡張生成)

RAG の正答率が低い。検索が悪いのか、LLM の読み取りが悪いのかを 1 問ずつ切り分けて、直すべき箇所を特定したい。

In [1]:
!pip install -q datasets rank_bm25 fugashi unidic-lite

In [2]:
import re
import unicodedata
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

ds = load_dataset("sbintuitions/JSQuAD", split="validation")
passages, pid, seen, eval_set = [], {}, set(), []
for ex in ds:
    c = ex["context"].split(" [SEP] ", 1)[-1]
    if c not in pid:
        pid[c] = len(passages); passages.append(c)
for ex in ds.shuffle(seed=0):
    c = ex["context"].split(" [SEP] ", 1)[-1]
    if pid[c] not in seen and len(eval_set) < 50:
        seen.add(pid[c]); eval_set.append({"q": ex["question"], "answers": sorted(set(ex["answers"]["text"])), "gold": pid[c]})

embedder = SentenceTransformer("intfloat/multilingual-e5-small")
p_emb = embedder.encode(["passage: " + p for p in passages], normalize_embeddings=True, batch_size=64, show_progress_bar=False)
ranking = np.argsort(-(embedder.encode(["query: " + e["q"] for e in eval_set], normalize_embeddings=True) @ p_emb.T), axis=1)

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

@torch.no_grad()
def generate(question, contexts):
    prompt = ("以下の資料だけに基づいて、質問に一文で短く答えてください。資料に答えが書かれていない場合は「わかりません」とだけ答えてください。\n\n"
              "# 資料\n" + "\n".join(f"- {c}" for c in contexts) + f"\n\n# 質問\n{question}")
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True)
    out = llm.generate(**inputs, max_new_tokens=60, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

norm = lambda s: re.sub(r"[\s、。「」『』()（）]", "", unicodedata.normalize("NFKC", s))
correct = lambda answer, golds: any(norm(g) in norm(answer) for g in golds)

In [3]:
K = 3
rows = []
for i, e in enumerate(eval_set):
    ids = ranking[i, :K].tolist()
    answer = generate(e["q"], [passages[j] for j in ids])
    ok, hit = correct(answer, e["answers"]), e["gold"] in ids
    if ok:
        category = "正答"
    elif not hit:
        category = "検索の失敗 (正解段落を渡せていない)"
    else:
        # 検索は当たっている。正解段落だけを渡せば答えられるかで、さらに切り分ける
        alone = generate(e["q"], [passages[e["gold"]]])
        category = "生成の失敗: 他の段落に惑わされた" if correct(alone, e["answers"]) else "生成の失敗: 正解段落だけでも答えられない"
    rows.append({"q": e["q"], "gold_rank": int(np.where(ranking[i] == e["gold"])[0][0]) + 1, "category": category, "answer": answer, "gold": e["answers"][0]})
df = pd.DataFrame(rows)
print(df["category"].value_counts().to_string())

category
正答                       41
生成の失敗: 正解段落だけでも答えられない     4
検索の失敗 (正解段落を渡せていない)       3
生成の失敗: 他の段落に惑わされた         2


In [4]:
pd.set_option("display.max_colwidth", 50)
for cat in df["category"].unique():
    if cat != "正答":
        print(f"--- {cat} ---")
        print(df[df["category"] == cat][["q", "gold_rank", "answer", "gold"]].head(3).to_string(index=False))

--- 生成の失敗: 正解段落だけでも答えられない ---
                 q  gold_rank                                              answer  gold
  PKFは通称、何と呼ばれている。          1                     PKFは通称、PKO（国際連合平和維持活動）と呼ばれています。 6章半活動
一番古い打楽器は手ともう一つは何か。          1                                   一番古い打楽器は手と棒や丸太です。     足
     民主主義革命の綱領の党は？          1 資料に記載されているラオス人民革命党が、民主主義革命の綱領において主に位置する党であると考えられます。 日本共産党
--- 生成の失敗: 他の段落に惑わされた ---
                        q  gold_rank                                                                                                                            answer gold
             ラオスで一番多い民族は?          1 資料に基づくと、ラオスで一番多い民族はラオ人であり、半数以上を占めています。しかし、ラオス政府はラオス国籍を持つ者を一様にラオス人として定義しているため、ラオ人を含む大多数の住民は統計上「ラオス人」とまとめられ、特定の「少数民族」としては分類されていません。 ラーオ族
共産党議員としての初の国会委員長となった者は誰か。          1                                                                                              共産党議員としての初の国会委員長となった者は、1979年に林百郎である。 田代文久
--- 検索の失敗 (正解段落を渡せていない) ---
               q  gold_rank                     

In [5]:
# 上限の確認: 検索が完璧だったら (正解段落だけを渡したら) 正答率はどこまで上がるか
oracle = np.mean([correct(generate(e["q"], [passages[e["gold"]]]), e["answers"]) for e in eval_set])
print(f"現在の正答率 {(df['category'] == '正答').mean():.2f} / 正解段落を必ず渡した場合 {oracle:.2f} / Recall@{K} = {(df['gold_rank'] <= K).mean():.2f}")

現在の正答率 0.82 / 正解段落を必ず渡した場合 0.82 / Recall@3 = 0.94


- 失敗を「検索の失敗」と「生成の失敗」に分けると、打ち手が決まる。検索の失敗が多ければ、ハイブリッド検索・リランキング・チャンクの見直し。生成の失敗が多ければ、プロンプト・渡す件数・モデルの見直し。
- 「正解段落だけを渡した場合の正答率」は、検索をどれだけ改善しても超えられない上限。これが低いなら、検索より先にモデルやプロンプトを直す。
- 正解段落の順位 (`gold_rank`) を見ると、「あと少しで入っていた」のか「まったく見当違い」なのかが分かる。前者は渡す件数を増やすかリランキングで拾える。
- 文字列照合で「誤答」とされたものには、言い換えただけの正答が混ざる。分類の前に、誤答の一覧を目で見て確かめる。